# Lab 14 — Deduplication and Data Mixing
**Goal:** Measure exact duplicate rates and simulate weighted sampling across languages/domains.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
texts=["hello world","hello world","machine learning","hello world","سلام دنیا","سلام دنیا"]
from collections import Counter
c=Counter(texts)
print(c)
print("exact duplicate rate:",1-len(set(texts))/len(texts))


## Prediction
Why can deduplication change training quality rather than merely storage size?


In [ ]:
import numpy as np
rng=np.random.default_rng(0)
langs=["en","fa","hi"]
weights=np.array([0.7,0.2,0.1])
sample=rng.choice(langs,size=10000,p=weights)
from collections import Counter
print(Counter(sample))



## Interpretation
Duplicates increase the effective sampling weight of repeated documents and can distort memorization and domain balance. Near-duplicate clusters can have similar effects.


In [ ]:
# Try different mixture weights and compare realized proportions.
for weights in [[.9,.05,.05],[.6,.3,.1],[.34,.33,.33]]:
    sample=rng.choice(langs,size=5000,p=weights)
    print(weights,{x:sample.tolist().count(x)/5000 for x in langs})


## Answers
Deduplication should be evaluated as a model-quality intervention. Mixing weights are also a modeling decision: they define which languages/domains receive learning capacity. Always test the mixture rather than treating it as purely administrative.

### Primary source
https://arxiv.org/abs/2402.00159


## Deepening: data interventions as controlled experiments

This section upgrades the notebook from a runnable demo to a measurable experiment. Predict before executing.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import math

mix = {"general":0.80, "domain":0.15, "low_resource":0.05}

def temperature_sample(p, alpha):
    w = {k: v**alpha for k,v in p.items()}
    z = sum(w.values())
    return {k: v/z for k,v in w.items()}

for alpha in [1.0, 0.75, 0.5, 0.25]:
    print("alpha", alpha, temperature_sample(mix, alpha))

docs = ["A","A","B","C","C","C","D"]
print("raw docs:", len(docs), "dedup docs:", len(list(dict.fromkeys(docs))))


## Break experiment

Compare separately:

1. stronger deduplication;
2. stronger low-resource sampling.

Do not change both at once.

Record token budget, source proportions, validation loss, target slice, and regression.

Use templates/EXPERIMENT_CARD.md to record the run.
